# Intro

This is a deep learning based Geo MLP model to predict city level crime risk from a mix of geographic coordinates and country level features. The goal is to estimate a crime index for any given latitude/longitude and country, even in places without direct city level data, by leveraging patterns in national crime indices, demographics, and population density. This first baseline model deliberately excludes city level safety scores to avoid trivial shortcuts, forcing the network to learn how far country context and location alone can go in explaining local risk.

In [1]:
# City Level Crime Index Predictor with Geo-MLP Model 


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
# Load data -----> lon/lat,safteyandcrimeindex are city level all other are country level   
 
city_path = "data/compiled_model_ready/MR_cities_with_country_v1.csv"
cities = pd.read_csv(city_path)

print("Columns:", cities.columns.tolist())
print(cities.head())
print("Number of cities:", len(cities))

Columns: ['city', 'country', 'country_norm', 'lat', 'lon', 'crime_index', 'safety_index', 'crimeindex_2023', 'crimeindex_2020', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per_km2']
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0         83.6          16.4            83.76            84.49   
1         82.0          18.0            76.86            77.49   
2         81.0          19.0            76.86            77.49   
3         80.7          19.3            80.79 

In [4]:
cities.shape

(509, 15)

In [5]:
#  Feature selection and target

feature_cols = [
    "lat",
    "lon",
    # core country risk signals
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    # demographics & scale
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
    
]

target_col = "crime_index"  

# Drop any rows missing required features or target 
cities_model = cities.dropna(subset=feature_cols + [target_col]).copy()

X = cities_model[feature_cols].values.astype(np.float32)
y = cities_model[target_col].values.astype(np.float32)

print("X shape:", X.shape, "y shape:", y.shape)

X shape: (509, 10) y shape: (509,)


In [6]:
# Train/test split and scaling

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)

Train: (407, 10) Test: (102, 10)


In [7]:
# Tensor datasets and DataLoaders

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [8]:
# Geo MLP model definition

class GeoMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),  # regression output
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train_scaled.shape[1]
model = GeoMLP(input_dim).to(device)
model

GeoMLP(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [9]:
# Training setup and loop

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

def evaluate(model, loader):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            out = model(xb)
            preds.append(out.cpu().numpy())
            targets.append(yb.cpu().numpy())
    preds = np.vstack(preds).ravel()
    targets = np.vstack(targets).ravel()
    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)
    return rmse, mae, r2

n_epochs = 400
best_rmse = float("inf")
best_state = None

for epoch in range(1, n_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

    if epoch % 20 == 0:
        train_rmse, train_mae, train_r2 = evaluate(model, train_loader)
        test_rmse, test_mae, test_r2 = evaluate(model, test_loader)
        print(
            f"Epoch {epoch:03d} | "
            f"Train RMSE {train_rmse:.2f}, R2 {train_r2:.3f} | "
            f"Test RMSE {test_rmse:.2f}, R2 {test_r2:.3f}"
        )
        if test_rmse < best_rmse:
            best_rmse = test_rmse
            best_state = model.state_dict()

print("Best test RMSE:", best_rmse)
if best_state is not None:
    model.load_state_dict(best_state)

Epoch 020 | Train RMSE 12.61, R2 0.313 | Test RMSE 14.98, R2 0.079
Epoch 040 | Train RMSE 10.91, R2 0.485 | Test RMSE 12.77, R2 0.330
Epoch 060 | Train RMSE 10.40, R2 0.532 | Test RMSE 12.13, R2 0.396
Epoch 080 | Train RMSE 10.20, R2 0.551 | Test RMSE 11.98, R2 0.411
Epoch 100 | Train RMSE 10.01, R2 0.567 | Test RMSE 11.70, R2 0.438
Epoch 120 | Train RMSE 9.91, R2 0.576 | Test RMSE 11.64, R2 0.444
Epoch 140 | Train RMSE 9.90, R2 0.577 | Test RMSE 11.61, R2 0.446
Epoch 160 | Train RMSE 9.80, R2 0.585 | Test RMSE 11.38, R2 0.468
Epoch 180 | Train RMSE 9.76, R2 0.588 | Test RMSE 11.39, R2 0.467
Epoch 200 | Train RMSE 9.74, R2 0.590 | Test RMSE 11.41, R2 0.465
Epoch 220 | Train RMSE 9.72, R2 0.591 | Test RMSE 11.40, R2 0.466
Epoch 240 | Train RMSE 9.72, R2 0.592 | Test RMSE 11.26, R2 0.479
Epoch 260 | Train RMSE 9.66, R2 0.596 | Test RMSE 11.33, R2 0.473
Epoch 280 | Train RMSE 9.65, R2 0.597 | Test RMSE 11.32, R2 0.473
Epoch 300 | Train RMSE 9.61, R2 0.601 | Test RMSE 11.27, R2 0.478
Epoch

In [10]:
# Final evaluation and example cities

test_rmse, test_mae, test_r2 = evaluate(model, test_loader)
print(f"Final Test — RMSE: {test_rmse:.2f}, MAE: {test_mae:.2f}, R2: {test_r2:.3f}")

# Inspect a few random cities
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(cities_model), size=5, replace=False)

for idx in sample_idx:
    row = cities_model.iloc[idx]
    x_raw = row[feature_cols].values.astype(np.float32).reshape(1, -1)
    x_scaled = scaler.transform(x_raw)
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        pred = model(x_tensor).cpu().numpy().ravel()[0]

    print("-" * 40)
    print(f"City: {row['city']} ({row['country']})")
    print(f"True crime_index: {row[target_col]:.1f}")
    print(f"Pred crime_index: {pred:.1f}")

Final Test — RMSE: 11.14, MAE: 8.75, R2: 0.491
----------------------------------------
City: Cluj-Napoca (Romania)
True crime_index: 22.5
Pred crime_index: 27.9
----------------------------------------
City: Chisinau (Moldova)
True crime_index: 43.9
Pred crime_index: 45.9
----------------------------------------
City: Shanghai (China)
True crime_index: 30.6
Pred crime_index: 30.0
----------------------------------------
City: Philadelphia (PA)
True crime_index: 64.9
Pred crime_index: 50.3
----------------------------------------
City: Bremen (Germany)
True crime_index: 44.3
Pred crime_index: 36.9


In [15]:
# Inference helper: (lat, lon, country) -> crime_index

country_path = "data/compiled_model_ready/country_features_2022_2023.csv"  
country_df = pd.read_csv(country_path)

# Normalize country name for matching
country_df["country_norm"] = country_df["country"].str.strip().str.lower()

def build_feature_vector_from_latlon(lat, lon, country_name):
    cname = country_name.strip().str.lower() if isinstance(country_name, pd._libs.missing.NAType) else str(country_name).strip().lower()
    row = country_df[country_df["country_norm"] == cname]
    if row.empty:
        raise ValueError(f"Country '{country_name}' not found in country feature table")
    r = row.iloc[0]

    # Build in the same order as feature_cols
    feat_vals = []
    for col in feature_cols:
        if col == "lat":
            feat_vals.append(float(lat))
        elif col == "lon":
            feat_vals.append(float(lon))
        else:
            feat_vals.append(float(r[col]))

    feat = np.array([feat_vals], dtype=np.float32)
    feat_scaled = scaler.transform(feat)
    return torch.tensor(feat_scaled, dtype=torch.float32)

def predict_crime_geomlp(lat, lon, country_name):
    model.eval()
    x = build_feature_vector_from_latlon(lat, lon, country_name).to(device)
    with torch.no_grad():
        pred = model(x).cpu().numpy().ravel()[0]
    return float(pred)


In [16]:
# Example:
example_pred = predict_crime_geomlp(-33.92, 18.42, "south africa")
print("Predicted crime_index near Cape Town:", example_pred)

Predicted crime_index near Cape Town: 79.64404296875


# shouldnt have to run again, if copied to another model version add features and change name to be model version specific  

In [12]:
# Build a country-level table from the cleaned city+country DF
country_cols = [
    "country",
    "country_norm",       
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

country_df = (
    cities[country_cols]
    .drop_duplicates(subset=["country_norm"])   #one row per country
    .reset_index(drop=True)
)

country_df.to_csv(
    "data/compiled_model_ready/country_features_2022_2023.csv",
    index=False
)

In [14]:
country_path = "data/compiled_model_ready/country_features_2022_2023.csv"
country_df = pd.read_csv(country_path)
country_df["country_norm"] = country_df["country_norm"].str.strip().str.lower()

# End Model --------------------------------------------------------Start Report

# Summary 

The initial results are genuinely encouraging with no city level safety feature, the model still achieves a test RMSE around 11 and captures roughly half of the variance in city crime, which is impressive given it’s only using country level context plus coordinates. Even more notable, this performance comes from the very first training run no architecture tuning, no hyperparameter search, and no feature engineering beyond the cleaned dataset. I’m going to snapshot this version as a baseline and then work on a tuned copy, where I can experiment with additional city level features, regularization, and architecture tweaks to push accuracy 

# Next Steps

The next phase is all about strengthening the city‑level signal and then tuning the Geo‑MLP on top of that richer feature space.

### Add more city‑level features

Add city‑specific population (even if partially manual), and from that derive simple city density (population / built‑up area or a rough proxy).

Incorporate city‑level pollution indicators where available (e.g., PM2.5, air quality index); these often correlate with urbanization and can proxy for environmental and economic conditions.

Consider simple urban form proxies like “is_capital”, “large_city_flag”, or size buckets (small/medium/large) derived from population.

### Introduce spatial/region encoding beyond raw lat/lon

Create a country code feature that is purely numeric: e.g., an integer ID, or a 2–3 digit code.

You can go further and define a grid code over each country: partition the country into a small grid and assign each city a cell ID; this lets the model learn within‑country regional patterns (north vs south, coastal vs inland) even before you add more complex spatial features.

Optionally one‑hot or embed these codes (especially the grid or regional codes) to help the model separate distinct zones.

### Expand environmental and contextual features

Add pollution data at the city level where possible, and consider augmenting with proxies like elevation, distance to coast, or distance to the capital (derived from lat/lon).

If you later pull in open data (e.g., POIs, nightlights, or land‑use), these can become powerful city‑level predictors layered onto the current framework.

### Hyperparameter and architecture tuning on a copied notebook

Clone this baseline notebook and experiment with:

Hidden layer sizes (e.g., 64→32 vs 128→64→32),

Dropout rates (0.1–0.4),

Weight decay and learning rates.

Use the same train/test split and log RMSE/R² for each run so you can compare against the current baseline and against KNN.

Benchmark against non DL baselines

Fit a strong tree based regressor (e.g., gradient boosting) on the same features/split and compare RMSE/R².

Use that as a reference to judge whether further DL tuning is giving real gains, or if most of the value is coming from feature quality rather than model class.

# DataSet Information

To build the dataset for the Geo‑MLP, I started by integrating five separate data sources into a single, city‑level table. This involved loading each dataset, standardizing column names, and joining them on shared keys such as country and city names. I carefully normalized country strings (lowercasing, trimming whitespace, handling variants like “United States” vs “USA”) to ensure a clean join, and I resolved duplicate or conflicting records so that each city ended up with one consistent row.

Next, I enriched the city table with geographic information by adding precise latitude and longitude coordinates for every location. This step makes the dataset “geo‑aware” and enables the model to learn spatial patterns in crime risk rather than relying solely on country averages. Alongside the coordinates, I merged in a comprehensive set of country‑level features for each city: crime and safety indices (for 2020 and 2023), demographic age splits (0–14, 15–64, 65+), total population, and population density per square kilometer. Where some countries were missing specific metrics (for example, 2020 crime or safety values for a few cases), I filled them using principled rules: copying 2023 crime into 2020 when that made sense, and, in rare edge cases, hard‑coding well‑researched safety values.

I then systematically identified and resolved missing values. Rather than blindly median‑filling, I inspected the small set of remaining NaNs city by city (e.g., San Juan, Montevideo, Tashkent, Kigali) and went back to external sources to look up accurate population, demographic, and density values. This meant manually inserting exact figures for those countries so the final dataset would be numerically complete without synthetic imputation. I also verified that each city row had a full, consistent set of country features and that no column still contained missing data before saving.

Throughout this process, I kept the data in a single, modeling‑ready dataframe: one row per city, with all city‑level fields (name, country, lat, lon, crime_index, safety_index if available) and all merged country‑level attributes. After final validation—checking dtypes, ranges, and confirming zero NaNs—I saved the cleaned table as cities_model_2022_2023_with_country.csv. This file is now a high‑quality foundation for training the Geo‑MLP, giving the model a rich combination of geographic coordinates, national risk context, and demographic structure for each city in the dataset.

In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path

# PDF 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_MLP_v1.ipynb to pdf
[NbConvertApp] Writing 59506 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 72904 bytes to Geo_MLP_v1.pdf
